# Section 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import html
from pathlib import Path

# Section 2. Define File Paths

In [4]:
INPUT_PATH = Path("postings.csv")

OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "IE7374 clean_postings.csv"

In [6]:
df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully.")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

Dataset loaded successfully.
Number of rows: 123,849
Number of columns: 31


# Section 4. Initial Data Exploration

In [9]:
pd.set_option("display.max_columns", None)

df.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,17.0,Full-time,2.0,1.713398e+12,NaN,https://www.linkedin.com/jobs/view/921716/?trk...,NaN,ComplexOnsiteApply,1.715990e+12,NaN,NaN,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,30.0,Full-time,NaN,1.712858e+12,NaN,https://www.linkedin.com/jobs/view/1829192/?tr...,NaN,ComplexOnsiteApply,1.715450e+12,NaN,NaN,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,45000.0,Full-time,NaN,1.713278e+12,NaN,https://www.linkedin.com/jobs/view/10998357/?t...,NaN,ComplexOnsiteApply,1.715870e+12,NaN,NaN,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,140000.0,Full-time,NaN,1.712896e+12,NaN,https://www.linkedin.com/jobs/view/23221523/?t...,NaN,ComplexOnsiteApply,1.715488e+12,NaN,NaN,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,60000.0,Full-time,NaN,1.713452e+12,NaN,https://www.linkedin.com/jobs/view/35982263/?t...,NaN,ComplexOnsiteApply,1.716044e+12,NaN,NaN,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [11]:
print(df.columns.tolist())

['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  str    
 2   title                       123849 non-null  str    
 3   description                 123842 non-null  str    
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   str    
 6   location                    123849 non-null  str    
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  str    
 12  applies                     23320 non-null   float64
 13  original_listed_time     

# Section 5. Missing Value Analysis

In [16]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

missing_summary = missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

missing_summary.head(31)

,missing_count,missing_percentage
closed_time,122776,99.133622
skills_desc,121410,98.030666
med_salary,117569,94.929309
remote_allowed,108603,87.689848
applies,100529,81.170619
min_salary,94056,75.944093
max_salary,94056,75.944093
currency,87776,70.873402
compensation_type,87776,70.873402
pay_period,87776,70.873402


Check important columns

In [19]:
important_columns = [
    "job_id",
    "title",
    "description",
    "company_name",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
    "skills_desc"
]

missing_summary.loc[important_columns]

,missing_count,missing_percentage
job_id,0,0.000000
title,0,0.000000
description,7,0.005652
company_name,1719,1.387981
location,0,0.000000
formatted_work_type,0,0.000000
formatted_experience_level,29409,23.745852
skills_desc,121410,98.030666


# Section 6. Duplicate Analysis

In [22]:
df["skills_desc"].notna().sum()

np.int64(2439)

In [24]:
full_duplicate_count = df.duplicated().sum()

print(f"Fully duplicated rows: {full_duplicate_count:,}")

Fully duplicated rows: 0


In [25]:
description_duplicate_count = df.duplicated(
    subset=["description"]
).sum()

print(f"Duplicated descriptions: {description_duplicate_count:,}")

Duplicated descriptions: 16,021


In [28]:
job_duplicate_count = df.duplicated(
    subset=["title", "description"]
).sum()

print(f"Duplicated title-description pairs: {job_duplicate_count:,}")

Duplicated title-description pairs: 12,944


# Section 7. Job Description Length Analysis

In [31]:
df["description_character_count"] = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.len()
)

df["description_word_count"] = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

In [32]:
df["description_word_count"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    123849.000000
mean        523.027929
std         301.940688
min           0.000000
1%           48.480000
5%          125.000000
10%         177.000000
25%         298.000000
50%         477.000000
75%         696.000000
90%         924.000000
95%        1074.000000
99%        1392.000000
max        3400.000000
Name: description_word_count, dtype: float64

View the shortest descriptions

In [36]:
df[
    ["title", "description", "description_word_count"]
].sort_values(
    by="description_word_count"
).head(20)

,title,description,description_word_count
96726,Territory Sales Manager,NaN,0
96992,Behavioral Health Technician,NaN,0
69767,Solar Energy Project Manager,NaN,0
20364,Promotional Products Sales Executive,NaN,0
70631,Head of Tax,NaN,0
55500,Asset Managers,NaN,0
22016,Director of Consulting & Insights,NaN,0
76063,CFAO,CFAO,1
89209,Senior Account Manager,https://cigna.wd5.myworkdayjobs.com/cignacaree...,1
34951,Service Specialist,https://www.jobs.virginia.gov/jobs/elections-a...,1


View the longest descriptions

In [39]:
df[
    ["title", "description", "description_word_count"]
].sort_values(
    by="description_word_count",
    ascending=False
).head(20)

,title,description,description_word_count
116608,Aircraft Mechanic - Parachute Rigger with Secu...,BASIC FUNCTION SUMMARY * Performs scheduled an...,3400
73603,Human Services Professional IV (Autism Spectru...,Description\n\nThis posting is being used to f...,3370
73730,"School Food Services Manager I, II - Kahaluu E...",Description\n\nThe authorized level of this po...,3220
110419,Sr. Network Engineer with Security Clearance,Title\n\nSr. Network Engineer KBR is actively ...,3090
98473,Park Ranger (Climbing),Summary\n\nThis position is located in Rocky M...,3077
96134,Logistics Management Specialist with Security ...,Duties * Establish management controls to ensu...,2964
7877,"Principal, Research (Human Resources)",About This Role\n\nWe are looking for a Resear...,2922
109838,"Physician, Family Medicine Physician",Job Description\n\nJob Description\n\nPark Nic...,2873
73761,"Educational Assistant I, II, III - Waianae High",Description\n\nThis posting is being used to f...,2857
73725,"Educational Assistant I, II, III - Heeia Eleme...",Description\n\nThis posting is being used to f...,2857


# Section 8. Experience Level Distribution

In [42]:
experience_distribution = (
    df["formatted_experience_level"]
    .value_counts(dropna=False)
    .to_frame(name="count")
)

experience_distribution["percentage"] = (
    experience_distribution["count"] / len(df) * 100
).round(2)

experience_distribution

,count,percentage
formatted_experience_level,,
Mid-Senior level,41489,33.50
Entry level,36708,29.64
NaN,29409,23.75
Associate,9826,7.93
Director,3746,3.02
Internship,1449,1.17
Executive,1222,0.99


# Section 9. Work Type Distribution

In [45]:
work_type_distribution = (
    df["formatted_work_type"]
    .value_counts(dropna=False)
    .to_frame(name="count")
)

work_type_distribution["percentage"] = (
    work_type_distribution["count"] / len(df) * 100
).round(2)

work_type_distribution

,count,percentage
formatted_work_type,,
Full-time,98814,79.79
Contract,12117,9.78
Part-time,9696,7.83
Temporary,1190,0.96
Internship,983,0.79
Volunteer,562,0.45
Other,487,0.39


# Section 10. Text Cleaning Function

In [48]:
def clean_text(value):
    """
    Clean a text field by removing HTML markup and normalizing whitespace.

    Parameters
    ----------
    value:
        Original text value.

    Returns
    -------
    str or np.nan:
        Cleaned text, or np.nan if the input is missing or empty.
    """
    
    if pd.isna(value):
        return np.nan

    text = str(value)

    # Convert HTML entities:
    # &amp; -> &
    # &nbsp; -> space
    text = html.unescape(text)

    # Replace common HTML line break tags with spaces
    text = re.sub(
        r"<\s*(br|p|div|li)\s*/?\s*>",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove all remaining HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Replace non-breaking spaces
    text = text.replace("\xa0", " ")

    # Replace line breaks and tabs with spaces
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Remove control characters
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    # Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove spaces at the beginning and end
    text = text.strip()

    if text == "":
        return np.nan

    return text

# Section 11. Data Cleaning

In [51]:
clean_df = df.copy()

In [53]:
initial_rows = len(clean_df)

print(f"Initial rows: {initial_rows:,}")

Initial rows: 123,849


Select useful columns

In [56]:
selected_columns = [
    "job_id",
    "company_name",
    "title",
    "description",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
]

clean_df = clean_df[selected_columns].copy()

Remove missing values

In [59]:
before = len(clean_df)

clean_df = clean_df.dropna(
    subset=["title", "description"]
).copy()

after = len(clean_df)

print(f"Removed missing title/description rows: {before - after:,}")

Removed missing title/description rows: 7


Clean text fields

In [62]:
text_columns = [
    "company_name",
    "title",
    "description",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
]

for column in text_columns:
    clean_df[column] = clean_df[column].apply(clean_text)

Remove duplicates

In [64]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["title", "description"],
    keep="first"
).copy()

after = len(clean_df)

print(f"Removed duplicated job postings: {before - after:,}")

Removed duplicated job postings: 13,168


Calculate description length

In [68]:
clean_df["description_character_count"] = (
    clean_df["description"].str.len()
)

clean_df["description_word_count"] = (
    clean_df["description"].str.split().str.len()
)

Filter short descriptions

In [71]:
MIN_DESCRIPTION_WORDS = 50

In [73]:
before = len(clean_df)

clean_df = clean_df[
    clean_df["description_word_count"] >= MIN_DESCRIPTION_WORDS
].copy()

after = len(clean_df)

print(
    f"Removed descriptions shorter than "
    f"{MIN_DESCRIPTION_WORDS} words: {before - after:,}"
)

Removed descriptions shorter than 50 words: 1,243


Fill missing values

In [76]:
optional_columns = [
    "company_name",
    "location",
    "formatted_work_type",
    "formatted_experience_level"
]

for column in optional_columns:
    clean_df[column] = clean_df[column].fillna("Not specified")

Reset index

In [79]:
clean_df = clean_df.reset_index(drop=True)

In [81]:
print(df.shape)
print(clean_df.shape)

(123849, 33)
(109431, 9)


# Section 12. Data Cleaning Summary

In [84]:
final_rows = len(clean_df)
removed_rows = initial_rows - final_rows
retention_rate = final_rows / initial_rows * 100

print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"Original rows:  {initial_rows:,}")
print(f"Final rows:     {final_rows:,}")
print(f"Removed rows:   {removed_rows:,}")
print(f"Retention rate: {retention_rate:.2f}%")

DATA CLEANING SUMMARY
Original rows:  123,849
Final rows:     109,431
Removed rows:   14,418
Retention rate: 88.36%


In [86]:
clean_df.head()

,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,description_character_count,description_word_count
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,Not specified,2525,358
1,1829192,Not specified,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,Not specified,3560,492
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",Full-time,Not specified,460,66
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",Full-time,Not specified,1594,209
4,91700727,Downtown Raleigh Alliance,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,"Raleigh, NC",Internship,Not specified,4188,578


In [88]:
clean_df.isna().sum()

job_id                         0
company_name                   0
title                          0
description                    0
location                       0
formatted_work_type            0
formatted_experience_level     0
description_character_count    0
description_word_count         0
dtype: int64

In [90]:
clean_df["description_word_count"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    109431.000000
mean        526.932286
std         302.588659
min          50.000000
1%           74.000000
5%          135.000000
25%         299.000000
50%         481.000000
75%         698.000000
95%        1079.000000
99%        1410.700000
max        3400.000000
Name: description_word_count, dtype: float64

In [92]:
clean_df.duplicated(
    subset=["title", "description"]
).sum()

np.int64(0)

# Section 13. Save Cleaned Dataset

In [95]:
clean_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")

Cleaned dataset saved to: processed_data\IE7374 clean_postings.csv


# Section 14. Generate Cleaning Report

In [97]:
cleaning_report = pd.DataFrame({
    "metric": [
        "original_rows",
        "final_rows",
        "removed_rows",
        "retention_rate_percentage",
        "minimum_description_words",
        "remaining_duplicate_jobs",
        "missing_titles",
        "missing_descriptions"
    ],
    "value": [
        initial_rows,
        final_rows,
        removed_rows,
        round(retention_rate, 2),
        MIN_DESCRIPTION_WORDS,
        clean_df.duplicated(
            subset=["title", "description"]
        ).sum(),
        clean_df["title"].isna().sum(),
        clean_df["description"].isna().sum()
    ]
})

cleaning_report

,metric,value
0,original_rows,123849.00
1,final_rows,109431.00
2,removed_rows,14418.00
3,retention_rate_percentage,88.36
4,minimum_description_words,50.00
5,remaining_duplicate_jobs,0.00
6,missing_titles,0.00
7,missing_descriptions,0.00


In [99]:
REPORT_PATH = OUTPUT_DIR / "IE7374 cleaning_report.csv"

cleaning_report.to_csv(
    REPORT_PATH,
    index=False
)

print(f"Cleaning report saved to: {REPORT_PATH}")

Cleaning report saved to: processed_data\IE7374 cleaning_report.csv
